# livecell — a tour

Everything below runs **locally**. Nothing is uploaded.

**Setup:** `deno jupyter --install`, then pick the **Deno** kernel (top right).

In [ ]:
// NOTE: the Deno kernel evaluates each cell as a SCRIPT, not a module — so
// `import.meta` does not exist here, and relative specifiers depend on the kernel's
// working directory. Locate the repo root explicitly, then import by absolute URL.

// `var` (not const/let) so re-running a cell doesn't throw "already declared".
function findUp(marker: string, from = Deno.cwd()): string {
  let dir = from;
  for (let i = 0; i < 8; i++) {
    try {
      Deno.statSync(`${dir}/${marker}`);
      return dir;
    } catch { /* keep walking up */ }
    const parent = dir.replace(/\/[^/]+$/, "");
    if (parent === dir) break;
    dir = parent;
  }
  throw new Error(`could not find ${marker} above ${from}`);
}

var root = findUp("deno.json");
var projectDir = `${root}/examples/demo-project`;

// Published: await import("jsr:@livecell/livecell")
var live = await import(`file://${root}/src/mod.ts`);

console.log("cwd    :", Deno.cwd());
console.log("root   :", root);
console.log("project:", projectDir);
console.log("port   :", live.start());

## 1 · Embed a real project

`mount` serves a folder; `embed` puts it in the cell. Click the button — it's a live page.

In [ ]:
live.mount("demo", projectDir);
live.embed("/m/demo/index.html", { height: 300, label: "demo project" });

## 2 · See what the page logged

`app.js` calls `console.log` on every click. livecell pipes that back into the notebook —
click the button above a few times, then run this cell.

In [ ]:
live.showLogs();          // styled block in the cell
console.log("as plain strings:", live.logs(5));   // or grab them as data
// live.clearLogs();      // reset between experiments

## 3 · Auto-reload while you edit

`watch` refreshes every embed when a file changes. Run this, then edit
`examples/demo-project/style.css` and watch the frame above update.

In [ ]:
live.watch(projectDir);
console.log("watching", projectDir, "— edit a file and the embeds reload");

## 4 · Diagrams that actually render

Notebook markdown cells **cannot** render Mermaid. Served from localhost, it just works.

In [ ]:
live.mermaid(`flowchart LR
  notes[concept notes] --> cell[runnable cell]
  cell --> app[live app on a port]
  app --> notes`, { height: 240 });

## 5 · Excalidraw — editable, and saved back to disk

`editable: true, save: true` turns the cell into a real editor. Move something, press
**Save**, and `demo-project/diagram.excalidraw.json` is rewritten.

In [ ]:
await live.excalidraw(`${projectDir}/diagram.excalidraw.json`, {
  editable: true,
  save: true,
  height: 420,
});

## 6 · A live playground

The only way a notebook can *show* how a layout behaves. Edit either pane.

In [ ]:
live.playground({
  html: `<div class="row">
  <div class="card">a</div>
  <div class="card">b</div>
  <div class="card">c</div>
</div>`,
  css: `.row { display: flex; justify-content: space-between; gap: 8px; }
.card { flex: 1; padding: 18px; background: #dbeafe; border-radius: 8px;
        font: 14px system-ui; text-align: center; cursor: pointer; }`,
  js: `document.querySelectorAll(".card").forEach((c, i) =>
  c.addEventListener("click", () => c.style.background = ["#fecaca","#bbf7d0","#fde68a"][i]));`,
  height: 420,
});

## 7 · Any server, any language

`serveCmd` spawns the process, waits until the port actually answers, then hands back the URL.
Works the same for `npm run dev`, `go run .`, `python3 -m http.server`.

In [ ]:
var url = await live.serveCmd("deno", [
  "eval",
  `Deno.serve({ port: 8931 }, () =>
     new Response("<body style='font:16px system-ui;padding:20px;background:#fff7ed'>" +
       "<h2 style='color:#c2410c'>Spawned server</h2><p>Started by a notebook cell.</p></body>",
       { headers: { "content-type": "text/html" } }));`,
], 8931);
live.embed(url, { height: 160, label: "deno server" });

## 8 · Any self-contained HTML, with real scripts

`page()` serves a string of HTML and hands back a URL. Because it comes from localhost
rather than an iframe `srcdoc`, **CDN scripts and ES modules load normally** — which is
exactly what VS Code's CSP blocks otherwise.

In [ ]:
const chart = live.page(`
  <body style="margin:0;font:14px system-ui">
    <canvas id="c" width="600" height="220"></canvas>
    <script type="module">
      // a real ES module import from a CDN — blocked in srcdoc, fine here
      const ctx = document.getElementById("c").getContext("2d");
      const data = [12, 40, 25, 60, 35, 80];
      data.forEach((v, i) => {
        ctx.fillStyle = "#0284c7";
        ctx.fillRect(20 + i * 95, 200 - v * 2, 60, v * 2);
      });
      ctx.fillStyle = "#0f172a";
      ctx.fillText("drawn by a module script inside the cell", 20, 215);
    </script>
  </body>`);
live.embed(chart, { height: 250, label: "custom page" });

## 9 · A snapshot for where the server doesn't exist

Embeds are **live only on this machine**. On GitHub, or in a colleague's checkout, an
iframe to `localhost` shows nothing.

`snapshot` captures a PNG so you have something durable. Reference it from a **markdown
cell** — notebook *outputs* are stripped from version control by `deno task setup`.

> First run downloads a headless Chromium (~45s). After that it takes a couple of seconds.

In [ ]:
const png = `${root}/examples/demo-project/preview.png`;
const saved = await live.snapshot(`${live.origin()}/m/demo/index.html`, png, {
  width: 900,
  height: 420,
});
console.log("saved:", saved);
console.log("reference it in markdown as:  ![preview](./demo-project/preview.png)");

Or let `embed` do both at once — live iframe now, PNG on disk for later:

```ts
live.embed("/m/demo/index.html", { height: 300, snapshot: `${root}/examples/preview.png` });

// await the file if a later step needs it to exist:
await live.embedWithSnapshot("/m/demo/index.html", { snapshot: png });
```

## 10 · Escape hatches

In [ ]:
// Are we actually in the kernel? (Deno.jupyter THROWS outside it, so this is guarded.)
console.log("in kernel:", live.inKernel());

// Emit any HTML you like as cell output.
live.html(`<div style="padding:10px;border-radius:6px;background:#ecfdf5;
  border:1px solid #10b981;font:14px system-ui">live.html() — arbitrary output</div>`);

// If a dev server's grandchildren outlive it (npm run dev spawning vite),
// free the port directly:
//   await live.killPort(5173);

## Tidy up

`stopAll` closes live-reload streams **first** (an open SSE stream would otherwise block
`server.shutdown()` forever), then stops watchers, then terminates children — escalating
`SIGTERM` → `SIGKILL` for anything that ignores it.

Pass `ports` to also free a port whose grandchildren outlived their parent
(`npm run dev` spawning vite, for example).

In [ ]:
console.log("before:", live.status());
console.log(await live.stopAll({ ports: [8931] }));
console.log("after :", live.status());